# Ordered Logistic Regression Results Dataset Exploration with `mlcroissant`
This notebook guides you through loading and exploring the "Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya" dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Install mlcroissant if not present
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`. This step fetches the Croissant schema and prepares the dataset for exploration.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)

# Access and print key metadata fields
metadata = dataset.metadata
print(f"Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Version: {metadata.version}\n")
print(f"License: {metadata.license}\n")
print(f"Temporal Coverage: {metadata.temporalCoverage}\n")
print(f"Spatial Coverage: {metadata.spatialCoverage}\n")
print(f"Data Collection: {metadata.dataCollection}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id` values using the Croissant schema. Every entity is referenced by its `@id`, providing unique identification across the dataset.

### Listing record sets and fields
The dataset may contain multiple record sets. Let's enumerate them and display their associated fields and columns (all by `@id`).

In [ ]:
# List available record sets by @id
record_sets = []
for rs in dataset.metadata.recordSet:
    print(f"RecordSet @id: {rs['@id']}, Name: {rs.get('name', 'N/A')}")
    record_sets.append(rs['@id'])
    # List fields by @id
    if 'field' in rs:
        for field in rs['field']:
            print(f"  Field @id: {field['@id']}, Name: {field.get('name', 'N/A')}, DataType: {field.get('dataType', 'N/A')}")
            # If column is defined, print column @id
            if 'column' in field:
                for col in field['column']:
                    print(f"    Column @id: {col['@id']}, Name: {col.get('name', 'N/A')}")
print("\nAvailable record sets:")
print(record_sets)

## 2a. Preview records from one record set
Let's select a sample record set by its `@id` and print a few sample records using `mlcroissant`. Replace `<record_set_id>` below with the correct `@id` found above.

In [ ]:
# Print preview records from a record set
# Example record_set_id, replace with actual @id from record_sets
if record_sets:
    example_record_set_id = record_sets[0]
    print(f"Preview records from RecordSet @id: {example_record_set_id}")
    for i, rec in enumerate(dataset.records(record_set=example_record_set_id)):
        print(rec)
        if i > 4:
            break
else:
    print('No record sets found.')

## 3. Data Extraction
Load data from each record set into a Pandas DataFrame for analysis. All entities are referenced by their `@id` per Croissant recommendations.

In [ ]:
# Load all available record sets as DataFrames
dataframes = {}

for record_set_id in record_sets:
    # Extract records for each record set
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"RecordSet @id: {record_set_id} Columns: {df.columns.tolist()}")
    print(df.head(2))

# Demonstrate main record set DataFrame preview
if record_sets:
    main_record_set = record_sets[0]
    print(f"\nPreview for main DataFrame (RecordSet @id: {main_record_set}):")
    print(dataframes[main_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, categorizing, removing outliers, transforming distributions, and grouping data by key attributes.

All references use the field and column `@id` for clarity and reproducibility.

In [ ]:
# EDA example: select a numeric field by @id
# Replace <numeric_field_id> and <group_field_id> with real @ids discovered earlier
main_df = dataframes[main_record_set]

# Example: Pick first numeric column from main_df
import numpy as np
numeric_field_id = None

# Try to auto-select a numeric field
for col in main_df.columns:
    if np.issubdtype(main_df[col].dtype, np.number):
        numeric_field_id = col
        break

if numeric_field_id:
    print(f"Using numeric field: {numeric_field_id}")
    threshold = main_df[numeric_field_id].mean()
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by another field (pick a categorical field)
    group_field_id = None
    for col in main_df.columns:
        if main_df[col].dtype == object and col != numeric_field_id:
            group_field_id = col
            break

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
    else:
        print("No categorical group field found.")
else:
    print("No numeric field found in the DataFrame.")

## 5. Visualization
Visualize selected numeric and grouped fields to understand the dataset distributions and relationships.
- Distribution of filtered numeric field
- Grouped mean by categorical field


In [ ]:
# Visualization with matplotlib
import matplotlib.pyplot as plt

if numeric_field_id:
    plt.figure(figsize=(6,4))
    filtered_df[numeric_field_id].hist(bins=20)
    plt.title(f"Distribution of filtered {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouped_df exists
    if 'grouped_df' in locals() and group_field_id:
        plt.figure(figsize=(8,4))
        plt.bar(grouped_df[group_field_id], grouped_df[numeric_field_id])
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook illustrates how to use the `mlcroissant` library for structured, reproducible exploration and processing of a complex dataset described by the Croissant schema.

Key steps included:
- Loading Croissant metadata and records by unique `@id`
- Listing available record sets, fields, and columns
- Extracting and preparing data for analysis
- Applying EDA for filtering, normalization, and grouping
- Data visualization for insights

You can use this workflow for scalable and rigorous data exploration of FAIR datasets. Extend the analysis for further statistical or modeling tasks as needed.